# Phase IVb — Scale hierarchy of artistic style

This notebook reuses the **4,000-image ArtBench-10 pilot feature matrix** from Phase IV. It does **not** re-download or re-extract images.

The scientific question is now more specific:

\[
\boxed{\text{At which spatial scales does geometric information generalize across unseen artists?}}
\]

The artist-level corpus previously showed strongest discrimination at fine scales, while coarse scales were much more resolution-stable. Here we test whether style-level generalization exhibits a different scale profile.

The primary evaluation remains **artist-disjoint nested cross-validation**. Every outer test fold contains artists absent from training; hyperparameters are selected only within the corresponding training fold.

Pre-specified contrasts:

1. geometry-only fine pair \(\sigma_{ref}=\{1,2\}\) vs coarse pair \(\{4,8\}\), matched at 20 features;
2. baseline+fine vs baseline+coarse, both selected to the same 90-dimensional budget;
3. scale-specific geometry increments over the same 90-feature conventional baseline;
4. repeat all analyses for all 10 styles and for the WikiArt-derived 8-style sensitivity subset.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Recover the Phase-IV pilot

Upload **`painting_geometry_phase4_artbench_pilot.zip`**. Only the already-computed feature matrix and prior Phase-IV summary tables are extracted. No image processing is repeated.

In [ ]:
from google.colab import files
import io, zipfile, pandas as pd, numpy as np

INPUT_DIR = Path("/content/phase4b_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = INPUT_DIR / "artbench_pilot_features.csv"

needed = {
    "artbench_pilot_features.csv",
    "artbench_artist_disjoint_results.csv",
    "artbench_artist_disjoint_deltas.csv",
    "phase4_run_metadata.json",
}

def extract_needed(blob: bytes):
    found = set()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for member in z.namelist():
            base = Path(member).name
            if base in needed:
                with z.open(member) as src, open(INPUT_DIR / base, "wb") as dst:
                    dst.write(src.read())
                found.add(base)
    return found

if not FEATURES.exists():
    print("Upload painting_geometry_phase4_artbench_pilot.zip")
    uploaded = files.upload()
    found = set()
    for name, blob in uploaded.items():
        if name.lower().endswith(".zip"):
            found |= extract_needed(blob)
    print("Recovered:", sorted(found))

if not FEATURES.exists():
    raise FileNotFoundError("artbench_pilot_features.csv was not found in the uploaded ZIP.")

feat = pd.read_csv(FEATURES)
print("Feature matrix:", feat.shape)
print("Styles:", feat["style"].nunique(), "Artists:", feat["artist"].nunique())
print("Baseline features:", sum(c.startswith("base__") for c in feat.columns))
print("Geometry features:", sum(c.startswith("geom__") for c in feat.columns))
display(feat.groupby("style")["artist"].nunique().to_frame("n_artists"))

## 2. Run scale hierarchy under unseen-artist evaluation

The geometry-only single-scale models each contain **10 descriptors**, so \(\sigma=1,2,4,8\) are dimensionally matched.

The fine/coarse comparison is also matched:

\[
G_{12}:20\text{ features},\qquad G_{48}:20\text{ features}.
\]

For complementarity, the conventional baseline has 90 features. Every baseline+geometry model is reduced **inside the training pipeline** to the same 90-feature budget with `SelectKBest`; therefore any scale-specific increment is not a simple dimensionality advantage.

In [ ]:
OUT = REPO_DIR / "results" / "phase4b_artbench_scale_hierarchy"
OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "scripts/run_artbench_scale_hierarchy.py",
    "--features", str(FEATURES),
    "--output-dir", str(OUT),
    "--matched-baseline-k", "90",
    "--outer-folds", "5",
    "--inner-folds", "3",
    "--n-jobs", "-1",
    "--metric-boot", "2000",
    "--delta-boot", "5000",
]
subprocess.run(cmd, check=True)

results = pd.read_csv(OUT / "phase4b_scale_hierarchy_results.csv")
deltas = pd.read_csv(OUT / "phase4b_scale_hierarchy_deltas.csv")
folds = pd.read_csv(OUT / "phase4b_scale_hierarchy_fold_results.csv")
per_style = pd.read_csv(OUT / "phase4b_scale_hierarchy_per_style.csv")

display(results)

## 3. Single-scale geometry: where does unseen-artist style information live?

In [ ]:
import matplotlib.pyplot as plt

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    sub = results[(results["dataset"] == dataset) & results["experiment"].isin(["G_s1", "G_s2", "G_s4", "G_s8"])].copy()
    order = ["G_s1", "G_s2", "G_s4", "G_s8"]
    sub["_order"] = sub["experiment"].map({x:i for i,x in enumerate(order)})
    sub = sub.sort_values("_order")
    x = np.array([1,2,4,8], dtype=float)
    y = sub["macro_f1_oof"].to_numpy()
    lo = sub["macro_f1_group_boot_ci_low"].to_numpy()
    hi = sub["macro_f1_group_boot_ci_high"].to_numpy()

    plt.figure(figsize=(7,4.5))
    plt.errorbar(x, y, yerr=[y-lo, hi-y], marker="o", capsize=4)
    plt.xscale("log", base=2)
    plt.xticks(x, ["1","2","4","8"])
    plt.xlabel(r"Reference Gaussian scale $\sigma_{ref}$")
    plt.ylabel("Artist-disjoint Macro-F1")
    plt.title(f"Single-scale level-set geometry — {dataset}")
    plt.tight_layout()
    path = OUT / f"Figure_{dataset}_single_scale_geometry.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

## 4. Fine vs coarse: the pre-specified hierarchy test

The central comparison is

\[
\Delta_{fine-coarse}=F_1(G_{1,2})-F_1(G_{4,8}).
\]

A positive interval entirely above zero means fine geometry generalizes better to unseen artists; a negative interval entirely below zero supports coarse-scale style organization. If the interval crosses zero, the data do not support a simple fine-vs-coarse hierarchy.

In [ ]:
fine_coarse = deltas[deltas["comparison"].isin([
    "geometry_fine_pair_vs_coarse_pair",
    "complementarity_fine_pair_vs_coarse_pair",
])].copy()
display(fine_coarse[[
    "dataset", "comparison", "delta_macro_f1", "delta_ci_low", "delta_ci_high", "bootstrap_p_new_gt_ref"
]])

## 5. Which scale contributes information beyond the conventional baseline?

In [ ]:
inc = deltas[deltas["comparison"].str.startswith("increment_scale_")].copy()
inc["sigma"] = inc["comparison"].str.extract(r"increment_scale_(\d+)_")[0].astype(float)

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    sub = inc[inc["dataset"] == dataset].sort_values("sigma")
    x = sub["sigma"].to_numpy()
    y = sub["delta_macro_f1"].to_numpy()
    lo = sub["delta_ci_low"].to_numpy()
    hi = sub["delta_ci_high"].to_numpy()

    plt.figure(figsize=(7,4.5))
    plt.errorbar(x, y, yerr=[y-lo, hi-y], marker="o", capsize=4)
    plt.axhline(0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xticks(x, [str(int(v)) for v in x])
    plt.xlabel(r"Reference Gaussian scale $\sigma_{ref}$")
    plt.ylabel(r"$\Delta$ Macro-F1 over 90-feature baseline")
    plt.title(f"Scale-specific complementary information — {dataset}")
    plt.tight_layout()
    path = OUT / f"Figure_{dataset}_scale_increment_over_baseline.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

display(inc[["dataset","sigma","delta_macro_f1","delta_ci_low","delta_ci_high","bootstrap_p_new_gt_ref"]])

## 6. Fold consistency and style-specific patterns

In [ ]:
focus = folds[folds["experiment"].isin(["G_s1","G_s2","G_s4","G_s8","G_fine_s12","G_coarse_s48"])].copy()
display(focus.pivot_table(index=["dataset","experiment"], columns="fold", values="macro_f1"))

# Per-style geometry comparison for descriptive interpretation only.
ps = per_style[per_style["experiment"].isin(["G_s1","G_s2","G_s4","G_s8"])].copy()
for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    table = ps[ps["dataset"] == dataset].pivot(index="style", columns="experiment", values="f1_one_vs_rest")
    display(table)

## 7. Compact scientific readout

Interpretation is deliberately conditional on the confidence intervals:

- **Fine > coarse** only if the 95% artist-group bootstrap CI for \(G_{12}-G_{48}\) is entirely positive.
- **Coarse > fine** only if that interval is entirely negative.
- Otherwise, no simple scale hierarchy is claimed.
- A scale is called **complementary** only when the CI for \(F_1(B+G_\sigma)-F_1(B)\) is entirely above zero under the matched 90-feature budget.
- The WikiArt-8 analysis is the stronger source-confound sensitivity check.

In [ ]:
def fmt(row):
    return f"Δ={row.delta_macro_f1:.4f}, 95% CI [{row.delta_ci_low:.4f}, {row.delta_ci_high:.4f}]"

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    print("\n===", dataset, "===")
    singles = results[(results["dataset"] == dataset) & results["experiment"].isin(["G_s1","G_s2","G_s4","G_s8"])].sort_values("macro_f1_oof", ascending=False)
    best = singles.iloc[0]
    print("Best single scale descriptively:", best["scale_set"], "Macro-F1=", round(best["macro_f1_oof"],4))

    fc = deltas[(deltas["dataset"] == dataset) & (deltas["comparison"] == "geometry_fine_pair_vs_coarse_pair")].iloc[0]
    print("Fine {1,2} - coarse {4,8}:", fmt(fc))

    comp = inc[inc["dataset"] == dataset].sort_values("delta_macro_f1", ascending=False)
    print("Largest single-scale increment over matched baseline:")
    print(comp[["sigma","delta_macro_f1","delta_ci_low","delta_ci_high"]].head(1).to_string(index=False))

## 8. Package outputs

Upload the resulting ZIP back to the analysis conversation. The next decision is whether the observed style-scale profile is strong enough to justify a larger ArtBench run, or whether the 4,000-image pilot already answers the hierarchy question.

In [ ]:
import json, shutil

PACKAGE = Path("/content/painting_geometry_phase4b_scale_hierarchy")
if PACKAGE.exists():
    shutil.rmtree(PACKAGE)
shutil.copytree(OUT, PACKAGE)

context = {
    "branch": BRANCH,
    "commit": subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip(),
    "input_phase4_features_shape": list(feat.shape),
    "primary_protocol": "artist_disjoint_nested_cv",
    "outer_folds": 5,
    "inner_folds": 3,
    "matched_baseline_dimension": 90,
    "single_scales": [1,2,4,8],
    "fine_pair": [1,2],
    "coarse_pair": [4,8],
}
(PACKAGE / "phase4b_run_context.json").write_text(json.dumps(context, indent=2), encoding="utf-8")

zip_path = shutil.make_archive("/content/painting_geometry_phase4b_scale_hierarchy", "zip", root_dir=PACKAGE)
print("Created:", zip_path)
files.download(zip_path)